# Day 58 — Code review, refactoring, and maintainability
Objectives:
- Refactor notebook code into reusable modules.
- Add unit tests and simple CI checklist.
- Improve documentation (docstrings, README sections).
- Prepare a maintainers checklist for DS repos.

## 1) From notebook to module
Take code from prior days (e.g., your preprocessing + training) and move it into a `src/` package.
Example layout:
````
ds-60day/
  src/
    __init__.py
    data.py        # load/clean functions
    features.py    # feature engineering
    model.py       # train/evaluate/save
  tests/
    test_model.py
  notebooks/
    ...
````


In [ ]:
# Example refactor target
from dataclasses import dataclass
from typing import Tuple
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

@dataclass
class TrainResult:
    pipeline: Pipeline
    auc: float

def build_pipeline() -> Pipeline:
    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['sex','class']),
                             ('num', StandardScaler(), ['fare','age'])])
    return Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])

def train_evaluate(df: pd.DataFrame) -> TrainResult:
    X = df[['sex','class','fare','age']]
    y = df['survived']
    Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
    pipe = build_pipeline()
    pipe.fit(Xtr,ytr)
    auc = roc_auc_score(yte, pipe.predict_proba(Xte)[:,1])
    return TrainResult(pipe, auc)


## 2) Tests with pytest
Create `tests/test_model.py` with unit tests for your functions.
Example:
```python
import pandas as pd
from src.model import train_evaluate

def test_train_evaluate_runs():
    df = pd.DataFrame([
        {
            'survived': i % 2,
            'sex': 'female' if i % 2 else 'male',
            'class': ('First', 'Second', 'Third')[i % 3],
            'fare': 20.0 + i,
            'age': 18.0 + (i % 30),
        }
        for i in range(40)
    ])
    res = train_evaluate(df)
    assert 0.5 <= res.auc <= 1.0
```
Run: `pytest -q`

## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — behavior-preserving refactoring, seams, tests, and compatibility review

### Mental model

Refactoring changes internal structure while preserving observable
behavior. Before changing unfamiliar code, a **characterization test**
records what it currently does for representative and boundary inputs.
Then small pure functions and explicit dependencies create seams that
can be tested without files, networks, clocks, or global state.

Code review is risk analysis, not style preference. Trace inputs,
outputs, exceptions, side effects, data/security boundaries, backward
compatibility, and evidence. Static tools catch classes of problems but
cannot prove runtime or domain behavior.

### Read the API before running it

- **characterization test:** locks down important current behavior before extracting or renaming it.
- **pure core + impure shell:** isolates transformations from I/O so normal, boundary, and failure cases are deterministic.
- **Ruff + mypy + pytest:** checks formatting/lint, static type contracts, and executable behavior as complementary evidence.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — capture legacy behavior before extraction

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Rounding only the final total is intended behavior; malformed-row policy still needs a failure test.

In [ ]:
def legacy_total(rows):
    return round(sum(float(row["price"]) * int(row["units"]) for row in rows), 2)

sample = [
    {"price": "1.25", "units": "2"},
    {"price": "0.50", "units": "3"},
]
observed = legacy_total(sample)
print(observed)
assert observed == 4.0  # characterization: preserve while restructuring

**Expected observation:** The test records the current numeric contract before parsing and calculation are separated.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — extract a pure boundary with explicit failures

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Rejecting negative values and translating errors matches the caller's contract.

In [ ]:
from decimal import Decimal, InvalidOperation

def line_total(price_text, units_text):
    try:
        price = Decimal(price_text)
        units = int(units_text)
    except (InvalidOperation, ValueError) as exc:
        raise ValueError("invalid sales row") from exc
    if price < 0 or units < 0:
        raise ValueError("price and units must be nonnegative")
    return price * units

assert line_total("1.25", "2") == Decimal("2.50")
try:
    line_total("bad", "2")
except ValueError as exc:
    print(type(exc.__cause__).__name__, str(exc))

**Expected observation:** The extracted function is deterministic, uses decimal arithmetic, and translates low-level parse errors into one boundary exception.

### Debugging and practice ramp

**Common mistake:** Changing behavior and structure simultaneously without a regression test or compatibility note.

**Diagnostic:** Reduce the diff, identify one observable contract, reproduce the old result, add boundary/failure tests, and run tools after each small extraction.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define behavior-preserving refactoring, seams, tests, and compatibility review in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not merge a refactor when tests assert implementation steps rather than behavior or when API/data compatibility is unexplained.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## 3) Document with docstrings & README
- Ensure every function has a concise docstring (inputs/outputs, assumptions).
- Add a short project README describing how to run training and tests.

## 4) Maintainability checklist
- Style and linting: Ruff; type contracts: mypy
- Tests pass: pytest
- Reproducibility: reviewed lock file, deterministic seeds
- Data contracts: pandera schemas on inputs/outputs
- Logging and error handling
- CI: run pytest + lint on PR

## Learner exercises and progressive hints

1. Move at least two notebook functions into a `src/` package and add tests.

**Verify:** Practice 1 — behavior-preserving refactoring, seams, tests, and compatibility review — import two moved functions from src in a fresh process and run focused normal/boundary/failure tests with exit code 0; prove the notebook now calls those imports and no copied implementation remains.

2. Add type hints and docstrings, then run mypy.

**Verify:** Practice 2 — behavior-preserving refactoring, seams, tests, and compatibility review — run mypy on the exact src/test paths with exit code 0 and save its transcript; include parameter/return annotations and docstrings that state errors/side effects, plus one negative typing fixture that fails as expected.

3. Write a short maintainer guide in the project root.

**Verify:** Practice 3 — behavior-preserving refactoring, seams, tests, and compatibility review — write a maintainer guide containing clean setup, test/lint/type commands, architecture/data flow, artifact locations, release/rollback, and troubleshooting; have another clean shell execute every command successfully.

### Progressive hints

1. Start with deterministic transformation/build functions. Create tiny
   synthetic DataFrames in fixtures so tests remain offline.
2. Type public boundaries first. A type-ignore requires a narrow reason;
   replacing every value with `Any` defeats the check.
3. Document setup, architecture, tests, formatting, data contracts, artifact
   locations, security rules, and review gates.

The separate solution demonstrates a small extraction, local pre-commit hooks,
and GitHub Actions. Treat CI as remote repetition of checks you can already run
locally.

### Additional mastery practice

Refactor behind tests, review risk at boundaries, and preserve behavior while improving structure. Type checks and formatting support—not replace—domain evidence.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Characterization testing:** Before refactoring a legacy notebook function, capture current behavior for normal, boundary, and known-bug inputs. Mark which behavior is a contract and which bug will intentionally change.
   **Progressive hint:** Characterization tests prevent accidental drift; an intentional fix needs a new expected result and a documented reason.

**Verify:** Characterization testing — save characterization tests and outputs for normal, boundary, and known-bug fixtures before editing; after refactor, assert contract cases match byte/value-for-value and the intentional bug change has a separately approved expected result.

5. **Risk-based review:** Review a data-loading-to-prediction change using a checklist for security, data loss, leakage, schema compatibility, performance, error handling, and cross-platform paths.
   **Progressive hint:** Trace inputs to side effects and downstream consumers. Prioritize high-impact boundaries over cosmetic preferences.

**Verify:** Risk-based review — complete a review matrix for security, data loss, leakage, schema, performance, errors, and Windows/POSIX paths; each row must cite changed lines, a test/measurement result, severity, owner, and disposition.

6. **Compatibility change:** Rename a public function parameter without breaking callers. Implement a deprecation path, tests for old/new usage, and a removal plan.
   **Progressive hint:** Accept the old keyword temporarily, reject ambiguous double use, emit a targeted DeprecationWarning, and update docs/call sites.

**Verify:** Compatibility change — assert old keyword and new keyword produce identical results during the compatibility window, simultaneous/conflicting use raises an error, the old path emits the named deprecation warning, and the removal version/date is documented.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Characterization testing


# Practice 5 — Risk-based review


# Practice 6 — Compatibility change
